In [16]:
## imports
import pandas as pd
import numpy as np
#import plotnine
#from plotnine import *
import random

## print multiple things from same cell
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from datetime import datetime, timedelta

## Load data

In [4]:
## load data on 2020 crimes in DC
df = dc_crim_2020 = pd.read_csv("https://opendata.arcgis.com/datasets/f516e0dd7b614b088ad781b0c4002331_2.csv")

## create report_dt column
df['report_dt'] = pd.to_datetime(df.REPORT_DAT)

## Warm-up Demo

In [5]:
%%time
for i in range(df.shape[0]):
    r = df.iloc[i]
    r.X + r.Y

CPU times: user 736 ms, sys: 6.51 ms, total: 743 ms
Wall time: 742 ms


In [6]:
%%time
for i,r in df.iterrows():
    r.X + r.Y

CPU times: user 466 ms, sys: 7.53 ms, total: 473 ms
Wall time: 472 ms


In [7]:
%%time
df.apply(lambda r: r.X + r.Y, axis = 1)

CPU times: user 178 ms, sys: 8.54 ms, total: 186 ms
Wall time: 185 ms


0        539842.0
1        535598.0
2        534219.0
3        534951.0
4        536599.0
           ...   
27927    535869.0
27928    530325.0
27929    537337.0
27930    539965.0
27931    536122.0
Length: 27932, dtype: float64

In [8]:
%%time
## Super fast, but only works with built-in numpy functions.
df.X + df.Y

CPU times: user 854 μs, sys: 438 μs, total: 1.29 ms
Wall time: 1.17 ms


0        539842.0
1        535598.0
2        534219.0
3        534951.0
4        536599.0
           ...   
27927    535869.0
27928    530325.0
27929    537337.0
27930    539965.0
27931    536122.0
Length: 27932, dtype: float64

# Practice

In [20]:
## define crimes to look for and crimes to look within
## CCN is Central Complaint Number: https://go.mpdconline.com/GO/GO_401_01.pdf
CCN_examples = ['20165648', '20123250']
C_Tar = C_Target = crimes_lookfor = df[df.CCN.astype(str).isin(CCN_examples)][['CCN', 'WARD', 'OFFENSE', 'report_dt']]
C_Oth = C_Other  = other_crimes = df[~df.CCN.astype(str).isin(CCN_examples)]

## print crimes_lookfor
C_Tar.head()
# other_crimes.head()


,CCN,WARD,OFFENSE,report_dt
4677,20165648,6,MOTOR VEHICLE THEFT,2020-11-20 02:25:50+00:00
21967,20123250,2,MOTOR VEHICLE THEFT,2020-08-29 05:00:25+00:00


**Task**: we have two crimes we want to look for. We want to look in the remaining crime reports for crime reports that are:

- Located in the same ward as the two focal crimes
- Reported at the same time as the focal crime or up to 1000 minutes later (changed from slides which stated 20 mins since crime ids changed since last time so this long bandwidth helps us find matches!)

Solutions compare two ways to solve:

- Using a for loop
- Using a function

## 1. Loop approach

In [17]:
## create empty container to store results 
store_matches = {}

## loop through two example crimes
for i in range(C_Tar.shape[0]): # same as shape
    
    ## extract row
    r = one_row = C_Tar.iloc[i]

    ## first, subset to crimes in same ward
    same_wards = C_Oth[C_Oth.WARD == r.WARD]
    
    ## second, with those same-ward crimes, construct indicator for reported within 20 minutes
    ## (interpreting as after but could do either)
    ### substep: get time cutoff
    CUTOFF = r.report_dt +  timedelta(minutes=1200)
    
    ### substep: use that to subset
    same_wards_sametime = same_wards[(same_wards.report_dt >= r.report_dt) & 
                                    (same_wards.report_dt <= CUTOFF)].copy()
    
    ## third, store the results
    store_matches[str(one_row.CCN)] = same_wards_sametime
    
## finally, concatenate results into one df
all_matches = pd.concat(store_matches)
all_matches.head()

X         Y       CCN              REPORT_DAT  \
20165648 2595   400232.0  135255.0  20165798  2020/11/20 12:46:32+00   
         2596   400233.0  137456.0  20165803  2020/11/20 14:45:06+00   
         5339   400489.0  136927.0  20165859  2020/11/20 15:37:59+00   
         8855   399489.0  137478.0  20165986  2020/11/20 22:17:27+00   
         10438  400042.0  135959.0  20165709  2020/11/20 04:27:36+00   

                            START_DATE                END_DATE  \
20165648 2595   2020/11/19 23:43:15+00                     NaN   
         2596   2020/11/19 23:45:48+00  2020/11/20 03:00:00+00   
         5339   2020/11/13 22:00:23+00  2020/11/14 00:00:13+00   
         8855   2020/11/20 20:15:26+00  2020/11/20 21:46:24+00   
         10438  2020/11/20 03:02:27+00                     NaN   

                                                    BLOCK  \
20165648 2595   600 - 669 BLOCK OF PENNSYLVANIA AVENUE SE   
         2596         600 - 699 BLOCK OF ORLEANS PLACE NE   
         5339              800 - 899 BLOCK OF H STREET NE   
         8855          1151 - 1199 BLOCK OF 1ST STREET NE   
         10438           100 - 199 BLOCK OF 5TH STREET NE   

                            OFFENSE  METHOD     SHIFT  ...  CENSUS_TRACT  \
20165648 2595           THEFT/OTHER  OTHERS       DAY  ...        6500.0   
         2596          THEFT F/AUTO  OTHERS       DAY  ...       10602.0   
         5339           THEFT/OTHER  OTHERS       DAY  ...        8402.0   
         8855   MOTOR VEHICLE THEFT  OTHERS   EVENING  ...       10603.0   
         10438  MOTOR VEHICLE THEFT  OTHERS  MIDNIGHT  ...        8200.0   

               VOTING_PRECINCT           BID    XBLOCK    YBLOCK   LATITUDE  \
20165648 2595      Precinct 89  CAPITOL HILL  400232.0  135255.0  38.885133   
         2596      Precinct 83           NaN  400233.0  137456.0  38.904961   
         5339      Precinct 82           NaN  400489.0  136927.0  38.900195   
         8855     Precinct 144          NOMA  399489.0  137478.0  38.905159   
         10438     Precinct 89           NaN  400042.0  135959.0  38.891475   

                LONGITUDE   OBJECTID OCTO_RECORD_ID                 report_dt  
20165648 2595  -76.997326  926272829            NaN 2020-11-20 12:46:32+00:00  
         2596  -76.997314  926272830            NaN 2020-11-20 14:45:06+00:00  
         5339  -76.994363  926304597            NaN 2020-11-20 15:37:59+00:00  
         8855  -77.005891  926316448            NaN 2020-11-20 22:17:27+00:00  
         10438 -76.999516  926323020            NaN 2020-11-20 04:27:36+00:00  

[5 rows x 26 columns]

# 1.5 Iterrow Approach

In [ ]:
## create empty container to store results 
store_matches = {}

## loop through two example crimes
for i, r in C_Tar.iterrows(): # same as 

    ## subset to crimes in same ward
    same_wards = C_Oth[C_Oth.WARD == r.WARD]
    
    ## second, with those same-ward crimes, construct indicator for reported within 20 minutes
    ## (interpreting as after but could do either)
    ### substep: get time cutoff
    CUTOFF = r.report_dt +  timedelta(minutes=1200)
    
    ### substep: use that to subset
    same_wards_sametime = same_wards[(same_wards.report_dt >= r.report_dt) & 
                                    (same_wards.report_dt <= CUTOFF)].copy()
    
    ## third, store the results
    store_matches[str(one_row.CCN)] = same_wards_sametime
    
## finally, concatenate results into one df
all_matches = pd.concat(store_matches)
all_matches.head()

## 2. Function approach

Practice rewriting the above loop as a function

### 2.1 define the function

In [44]:
store_matches_2 = {}

def find_related_crimes(r): # imagine the function taking in one row as its sole variable
    r = one_row = C_Tar.iloc[i]
    ## subset to crimes in same ward
    same_wards = C_Oth[C_Oth.WARD == r.WARD]
    
    ## second, with those same-ward crimes, construct indicator for reported within 20 minutes
    ## (interpreting as after but could do either)
    ### substep: get time cutoff
    CUTOFF = r.report_dt +  timedelta(minutes=1200)

    ### substep: use that to subset
    same_wards_sametime = same_wards[(same_wards.report_dt >= r.report_dt) & 
                                    (same_wards.report_dt <= CUTOFF)].copy()

    return same_wards_sametime


### 2.2 apply it to one of the focal crimes

In [46]:
C_Tar

,CCN,WARD,OFFENSE,report_dt
4677,20165648,6,MOTOR VEHICLE THEFT,2020-11-20 02:25:50+00:00
21967,20123250,2,MOTOR VEHICLE THEFT,2020-08-29 05:00:25+00:00


In [47]:
for i in range(C_Tar.shape[0]):
    store_matches_2[i] = find_related_crimes(r)
store_matches_2 ## i did this, not sure if correct. in class he showed:

{0:               X         Y       CCN              REPORT_DAT  \
 2595   400232.0  135255.0  20165798  2020/11/20 12:46:32+00   
 2596   400233.0  137456.0  20165803  2020/11/20 14:45:06+00   
 5339   400489.0  136927.0  20165859  2020/11/20 15:37:59+00   
 8855   399489.0  137478.0  20165986  2020/11/20 22:17:27+00   
 10438  400042.0  135959.0  20165709  2020/11/20 04:27:36+00   
 12130  400233.0  137456.0  20165805  2020/11/20 15:06:04+00   
 22157  399886.0  136784.0  20165932  2020/11/20 18:56:18+00   
 22159  398651.0  136899.0  20166039  2020/11/20 22:07:10+00   
 
                    START_DATE                END_DATE  \
 2595   2020/11/19 23:43:15+00                     NaN   
 2596   2020/11/19 23:45:48+00  2020/11/20 03:00:00+00   
 5339   2020/11/13 22:00:23+00  2020/11/14 00:00:13+00   
 8855   2020/11/20 20:15:26+00  2020/11/20 21:46:24+00   
 10438  2020/11/20 03:02:27+00                     NaN   
 12130  2020/11/19 22:30:39+00  2020/11/20 03:00:43+00   
 22157  2020/

In [48]:
pd.concat( store_matches_2 )

X         Y       CCN              REPORT_DAT  \
0 2595   400232.0  135255.0  20165798  2020/11/20 12:46:32+00   
  2596   400233.0  137456.0  20165803  2020/11/20 14:45:06+00   
  5339   400489.0  136927.0  20165859  2020/11/20 15:37:59+00   
  8855   399489.0  137478.0  20165986  2020/11/20 22:17:27+00   
  10438  400042.0  135959.0  20165709  2020/11/20 04:27:36+00   
  12130  400233.0  137456.0  20165805  2020/11/20 15:06:04+00   
  22157  399886.0  136784.0  20165932  2020/11/20 18:56:18+00   
  22159  398651.0  136899.0  20166039  2020/11/20 22:07:10+00   
1 4598   395618.0  138388.0  20123422  2020/08/29 16:45:57+00   
  4843   396662.0  138429.0  20401318  2020/08/29 14:29:59+00   
  5806   398098.0  136808.0  20123419  2020/08/29 17:15:19+00   
  11678  397609.0  136611.0  20123609  2020/08/30 00:05:52+00   
  14065  396546.0  137533.0  20123507  2020/08/29 22:04:46+00   
  16014  396523.0  137976.0  20123389  2020/08/29 16:05:18+00   

                     START_DATE                END_DATE  \
0 2595   2020/11/19 23:43:15+00                     NaN   
  2596   2020/11/19 23:45:48+00  2020/11/20 03:00:00+00   
  5339   2020/11/13 22:00:23+00  2020/11/14 00:00:13+00   
  8855   2020/11/20 20:15:26+00  2020/11/20 21:46:24+00   
  10438  2020/11/20 03:02:27+00                     NaN   
  12130  2020/11/19 22:30:39+00  2020/11/20 03:00:43+00   
  22157  2020/11/20 15:30:02+00  2020/11/20 18:25:35+00   
  22159  2020/11/20 17:30:16+00  2020/11/20 22:08:28+00   
1 4598   2020/08/26 22:00:29+00  2020/08/27 12:00:51+00   
  4843   2020/08/28 20:55:00+00  2020/08/28 21:05:00+00   
  5806   2020/08/29 16:05:40+00  2020/08/29 16:08:33+00   
  11678  2020/08/29 23:08:57+00                     NaN   
  14065  2020/08/27 19:01:24+00  2020/08/29 19:00:05+00   
  16014  2020/08/28 22:00:23+00  2020/08/29 08:00:27+00   

                                              BLOCK              OFFENSE  \
0 2595    600 - 669 BLOCK OF PENNSYLVANIA AVENUE SE          THEFT/OTHER   
  2596          600 - 699 BLOCK OF ORLEANS PLACE NE         THEFT F/AUTO   
  5339               800 - 899 BLOCK OF H STREET NE          THEFT/OTHER   
  8855           1151 - 1199 BLOCK OF 1ST STREET NE  MOTOR VEHICLE THEFT   
  10438            100 - 199 BLOCK OF 5TH STREET NE  MOTOR VEHICLE THEFT   
  12130         600 - 699 BLOCK OF ORLEANS PLACE NE         THEFT F/AUTO   
  22157              300 - 399 BLOCK OF G STREET NE         THEFT F/AUTO   
  22159  300 - 363 BLOCK OF MASSACHUSETTS AVENUE NW          THEFT/OTHER   
1 4598        2200 - 2399 BLOCK OF DECATUR PLACE NW         THEFT F/AUTO   
  4843          1724 - 1799 BLOCK OF 17TH STREET NW          THEFT/OTHER   
  5806             700 - 799 BLOCK OF 7TH STREET NW          THEFT/OTHER   
  11678            1100 - 1199 BLOCK OF F STREET NW  MOTOR VEHICLE THEFT   
  14065            1700 - 1779 BLOCK OF M STREET NW  MOTOR VEHICLE THEFT   
  16014            1700 - 1799 BLOCK OF P STREET NW         THEFT F/AUTO   

         METHOD     SHIFT  ...  CENSUS_TRACT VOTING_PRECINCT              BID  \
0 2595   OTHERS       DAY  ...        6500.0     Precinct 89     CAPITOL HILL   
  2596   OTHERS       DAY  ...       10602.0     Precinct 83              NaN   
  5339   OTHERS       DAY  ...        8402.0     Precinct 82              NaN   
  8855   OTHERS   EVENING  ...       10603.0    Precinct 144             NOMA   
  10438  OTHERS  MIDNIGHT  ...        8200.0     Precinct 89              NaN   
  12130  OTHERS       DAY  ...       10602.0     Precinct 83              NaN   
  22157  OTHERS       DAY  ...        8301.0     Precinct 83              NaN   
  22159  OTHERS   EVENING  ...        5900.0    Precinct 143         DOWNTOWN   
1 4598   OTHERS       DAY  ...        4100.0     Precinct 13              NaN   
  4843   OTHERS       DAY  ...        5302.0     Precinct 15              NaN   
  5806   OTHERS       DAY  ...        5801.0    Precinct 129         DOWNTOWN   
  11678  OTHERS   EVENING  ...        580

### 2.3 Use apply to cover all the other focal crimes

In [31]:
C_Tar.apply(find_related_crimes, axis=1)

4677                   X         Y       CCN           ...
21967                  X         Y       CCN           ...
dtype: object